# Model Explainability using SHAP

## Objective

The objective of this notebook is to explain the predictions generated by the Isolation Forest model using SHAP (SHapley Additive exPlanations).

Explainability improves transparency by identifying how individual features influence anomaly detection decisions, enabling users to better understand why specific observations are classified as anomalous.

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Explainable AI

Machine learning models often behave as "black boxes," making it difficult to understand the reasoning behind their predictions.

Explainable Artificial Intelligence (XAI) addresses this challenge by providing interpretable explanations that reveal the contribution of each feature toward a model's decision.

In [ ]:
import joblib

best_iso_model = joblib.load("best_isolation_forest_model.pkl")
scaler = joblib.load("standard_scaler.pkl")


## Input Data

The dataset used in this notebook contains engineered features and anomaly predictions generated during previous stages of the EcoWatt AI pipeline.

These features are supplied to the trained Isolation Forest model to compute SHAP values and interpret anomaly predictions.

In [ ]:
evaluation_df=pd.read_csv("data/processed/final_energy_anomaly_results.csv")

In [ ]:
feature_columns = [

    # Original electrical variables
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    # Temporal
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "year_sin",
    "year_cos",

    # Lag
    "active_power_lag_1",
    "active_power_lag_5",
    "active_power_lag_15",
    "active_power_lag_60",

    # Rolling
    "active_power_rolling_mean_15",
    "active_power_rolling_std_15",
    "active_power_rolling_min_15",
    "active_power_rolling_max_15",

    "active_power_rolling_mean_60",
    "active_power_rolling_std_60",
    "active_power_rolling_min_60",
    "active_power_rolling_max_60",

    # Behaviour
    "active_power_change_1",
    "deviation_from_15min_mean",
    "deviation_from_60min_mean",

    # Log transformed
    "active_power_change_rate_log",
    "rolling_zscore_15_log",
    "rolling_zscore_60_log"
]

In [ ]:
X = evaluation_df[feature_columns]
X_scaled = scaler.transform(X)

## Why SHAP?

SHAP (SHapley Additive exPlanations) is a model-agnostic explainability framework based on cooperative game theory.

Each feature receives a contribution value that represents its impact on the model's prediction.

Positive and negative SHAP values indicate how features influence the anomaly score.

In [ ]:
print(shap.__version__)

In [ ]:
# Background data for SHAP
background = shap.sample(X_scaled, 100, random_state=42)

# Data to explain
explain_data = X_scaled[:100]

## SHAP Explainer

The SHAP explainer estimates the contribution of every feature by comparing model predictions with and without that feature.

For Isolation Forest, SHAP explains how each engineered feature contributes to identifying abnormal energy consumption.

In [ ]:
explainer = shap.KernelExplainer(
    best_iso_model.decision_function,
    background
)

## Computing SHAP Values

SHAP values are calculated for selected observations.

These values quantify the importance of every feature for each prediction, allowing local explanations of individual anomalies.

In [ ]:
shap_values = explainer.shap_values(explain_data)

In [ ]:
anomaly_index = evaluation_df[
    evaluation_df["predicted_anomaly"] == 1
].index[0]

print("Anomaly Index:", anomaly_index)

## Force Plot

The SHAP Force Plot provides an intuitive visualization of how individual features interact to produce the final prediction.

Features pushing the prediction toward an anomaly are contrasted with those supporting normal behaviour.

In [ ]:
shap.force_plot(
    explainer.expected_value,
    shap_values[anomaly_index % 100],
    explain_data[anomaly_index % 100],
    feature_names=feature_columns,
    matplotlib=True
)

In [ ]:
anomaly_indices = evaluation_df[
    evaluation_df["predicted_anomaly"] == 1
].index

print("Number of anomalies:", len(anomaly_indices))

sample_index = anomaly_indices[0]

print("Selected anomaly index:", sample_index)

## Local Explanation

Unlike global feature importance, the Waterfall Plot explains a single prediction.

It illustrates how individual feature values push the prediction toward either normal behaviour or anomalous behaviour.

This enables detailed investigation of specific energy consumption events.

In [ ]:
sample_position = sample_index % 100

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_position],
        base_values=explainer.expected_value,
        data=explain_data[sample_position],
        feature_names=feature_columns
    )
)

## Global Feature Importance

The SHAP Summary Plot provides a global view of feature importance across the dataset.

Features appearing near the top have the greatest influence on anomaly detection.

This visualization highlights which characteristics of household electricity consumption are most informative.

In [ ]:
plt.figure(figsize=(12, 8))

shap.summary_plot(
    shap_values,
    explain_data,
    feature_names=feature_columns,
    show=False
)

plt.tight_layout()
plt.savefig("results/shap_summary_plot.png", dpi=300)
plt.show()

## Average Feature Contribution

The SHAP Bar Plot ranks features according to their average absolute SHAP values.

Higher values indicate stronger overall influence on the Isolation Forest model.

This ranking helps identify the most important engineered features.

In [ ]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    explain_data,
    feature_names=feature_columns,
    plot_type="bar",
    show=False
)

plt.tight_layout()
plt.savefig("results/shap_feature_importance.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Mean |SHAP| Value": np.abs(shap_values).mean(axis=0)
})

feature_importance = feature_importance.sort_values(
    "Mean |SHAP| Value",
    ascending=False
)

feature_importance.head(20)

In [ ]:
top10_features = feature_importance.head(10)

print(top10_features)

In [ ]:
report_table = feature_importance.head(10).copy()

report_table.columns = [
    "Feature",
    "Importance Score"
]

report_table

## Interpretation Summary

The SHAP analysis transforms the Isolation Forest model from a black-box algorithm into an interpretable machine learning system.

Users can understand not only which observations are anomalous but also the specific features responsible for those predictions.

In [ ]:
report_table.to_csv(
    "C:/EcoWatt-AI/data/processed/top10_feature_importance.csv",
    index=False
)

# Benefits of Explainable AI

Applying SHAP provides several important advantages:

- Improves model transparency.
- Builds user trust in AI predictions.
- Supports debugging and model validation.
- Identifies influential engineered features.
- Facilitates decision-making based on interpretable insights.
- Enhances deployment in real-world energy monitoring applications.

# Conclusion

This notebook successfully applied SHAP to interpret the predictions generated by the Isolation Forest model.

The explainability framework revealed both global feature importance and local feature contributions for individual anomaly predictions.

By combining anomaly detection with explainable AI, the EcoWatt AI system provides transparent, interpretable, and trustworthy insights into household electricity consumption.

The complete project now consists of an end-to-end machine learning pipeline including preprocessing, feature engineering, anomaly detection, model evaluation, explainability, and interactive dashboard deployment.